# Homework: Constraint Satisfaction Problems

Students implement Backtracking Search to solve the graph coloring problem. \
Students can add supporting attributes and methods to related classes as needed.

# Libraries

In [1]:
import os
import heapq

# Graph class

In [2]:
# Directed, weighted graphs
class Graph:
  def __init__(self):
    self.AL = dict() # adjacency list
    self.V = 0
    self.E = 0

  def __str__(self):
    res = 'V: %d, E: %d\n'%(self.V, self.E)
    for u, neighbors in self.AL.items():
      line = '%d: %s\n'%(u, str(neighbors))
      res += line
    return res

  def print(self):
    print(str(self))

  def load_from_file(self, filename):
    '''
        Example input file:
            V E
            u v w
            u v w
            u v w
            ...
            u1 h1
            u2 h2
            u3 h3
            ...

        # input.txt
        7 8
        0	1 1
        0	2 1
        1	2 1
        1	3 1
        2	4 1
        3	4 1
        4	5 1
        5	6 1
    '''
    if os.path.exists(filename):
      with open(filename) as g:
        self.V, self.E = [int(it) for it in g.readline().split()]
        for i in range(self.E):
          line = g.readline()
          u, v, w = [int(it) for it in line.strip().split()]
          if u not in self.AL:
            self.AL[u] = []
          self.AL[u].append((v, w))

In [3]:
g = Graph()
g.load_from_file('input.txt')
g.print()

V: 7, E: 8
0: [(1, 1), (2, 1)]
1: [(2, 1), (3, 1)]
2: [(4, 1)]
3: [(4, 1)]
4: [(5, 1)]
5: [(6, 1)]



# Search Strategies

In [4]:
class BackTrackingSearch:
  def __init__(self):
    self.colors = ['red', 'green', 'blue']
    self.neighbors = dict()
    self.variables = []

  def _prepare_csp(self, csp: Graph):
    nodes = set(range(csp.V))
    for u, adj in csp.AL.items():
      nodes.add(u)
      for v, _ in adj:
        nodes.add(v)

    self.variables = sorted(list(nodes))
    self.neighbors = {u: set() for u in self.variables}

    for u, adj in csp.AL.items():
      for v, _ in adj:
        if u != v:
          self.neighbors[u].add(v)
          self.neighbors[v].add(u)

  def is_consistent(self, var: int, value: str, assignment: dict) -> bool:
    for neighbor in self.neighbors[var]:
      if neighbor in assignment and assignment[neighbor] == value:
        return False
    return True

  def select_unassigned_variable(self, assignment: dict, domains: dict) -> int:
    unassigned = [v for v in self.variables if v not in assignment]
    return min(
        unassigned,
        key=lambda var: (
            len(domains[var]),
            -sum(1 for nb in self.neighbors[var] if nb not in assignment)
        )
    )

  def order_domain_values(self, var: int, assignment: dict, domains: dict) -> list:
    def count_conflicts(value):
      conflicts = 0
      for nb in self.neighbors[var]:
        if nb not in assignment and value in domains[nb]:
          conflicts += 1
      return conflicts

    return sorted(domains[var], key=count_conflicts)

  def inference(self, var: int, value: str, assignment: dict, domains: dict):
    pruned = []
    for nb in self.neighbors[var]:
      if nb not in assignment and value in domains[nb]:
        domains[nb].remove(value)
        pruned.append((nb, value))
        if len(domains[nb]) == 0:
          return None  
    return pruned

  def backtrack(self, assignment: dict, domains: dict):
    if len(assignment) == len(self.variables):
      return assignment

    var = self.select_unassigned_variable(assignment, domains)

    for value in self.order_domain_values(var, assignment, domains):
      if self.is_consistent(var, value, assignment):
        assignment[var] = value

        pruned = self.inference(var, value, assignment, domains)
        if pruned is not None:
          result = self.backtrack(assignment, domains)
          if result is not None:
            return result

        if pruned is not None:
          for nb, val in pruned:
            domains[nb].append(val)
        del assignment[var]

    return None

  def search(self, csp: Graph) -> dict:
    '''
    return a dict {node : color} in which
      node: a node in the given graph
      color: the assigned color, {red, green, blue}
    '''
    if csp.V == 0 and not csp.AL:
      return dict()

    self._prepare_csp(csp)
    domains = {var: list(self.colors) for var in self.variables}
    assignment = dict()

    result = self.backtrack(assignment, domains)
    return result if result is not None else dict()

# Evaluation

In [5]:
colors = BackTrackingSearch().search(g)
for node, color in sorted(colors.items()):
  print(node, color)

0 blue
1 red
2 green
3 green
4 red
5 green
6 red


# Submission

*   Students download the notebook after completion
*   Rename the notebook in which inserting your student ID at the beginning. \
For example, **123456-lec08-CSPs-HW.ipynb**
*   Finally, submit the file